# T3 — Model Registry: registro, promoção e rollback

**Objetivo:** governar versões de modelo. Quem está em produção? Quem aprovou? Como voltar atrás?

| Conceito | O que é |
|---|---|
| **Registered Model** | o "produto" (`bike-demand`) |
| **Version** | cada modelo registrado ganha um número: 1, 2, 3… (imutável) |
| **Alias** | ponteiro móvel para uma versão: `@champion` (produção), `@challenger` (candidato) |
| **Tags** | metadados da versão: `validation_status`, `previous_champion`… |

> Em material antigo você verá *stages* (`Staging`, `Production`) e `transition_model_version_stage`.
> Eles estão **descontinuados desde o MLflow 2.9**. O jeito atual é **aliases + tags**.

**Dois caminhos.** Registrar, descrever, marcar com tags e mover aliases também dá para fazer na interface do MLflow.
Em cada passo vem primeiro o código e, logo abaixo, o caminho pela interface (🖱️). Na Parte 1 rodamos o código
e usamos a interface para conferir o resultado. Na Parte 2, você escolhe o caminho na promoção.

> 🖱️ Na interface, o **Model registry** aparece no menu da esquerda quando o modo **Model training**
> está selecionado (botões no topo do menu).

In [1]:
import os, sys
from pathlib import Path

# Trabalha a partir da raiz do repositório (a pasta que contém o params.yaml)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "params.yaml").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import mlflow
import pandas as pd
from IPython.display import display
from mlflow import MlflowClient
from bike import MODEL_NAME, load_raw, regression_metrics, time_split, tracking_uri, xy

pd.set_option("display.width", 160)
mlflow.set_tracking_uri(tracking_uri())
client = MlflowClient()

_, test = time_split(load_raw(), test_days=28)
X_test, y_test = xy(test)


def avaliar(alias):
    """Carrega o modelo pelo alias e mede no MESMO teste para todos."""
    model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@{alias}")
    return regression_metrics(y_test, model.predict(X_test))


def prever_com_champion(X):
    """O 'consumidor' (a API da Aula 2): só conhece o alias, nunca o número da versão."""
    model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
    return model.predict(X).round(0)


def resumo_registry():
    aliases = client.get_registered_model(MODEL_NAME).aliases   # {alias: versão}
    linhas = []
    for mv in client.search_model_versions(f"name='{MODEL_NAME}'"):
        linhas.append({
            "versão": int(mv.version),
            "aliases": ", ".join(a for a, v in aliases.items() if v == mv.version),
            "origem": mv.tags.get("source_run_name", "pipeline (T1)"),
            "autor": mv.tags.get("autor", ""),
            "status": mv.tags.get("validation_status", ""),
            "descrição": mv.description,
        })
    return pd.DataFrame(linhas).sort_values("versão")

/home/vscode/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
# Parte 1 · Juntos

## 1. Escolher candidatos entre os runs da turma (T2)

In [2]:
runs = mlflow.search_runs(experiment_names=["aula01-experimentos"],
                          filter_string="tags.autor = 'turma' and metrics.rmse > 0",
                          order_by=["metrics.rmse ASC"])
baseline = runs[runs["tags.mlflow.runName"] == "baseline-linear"].iloc[0]
melhor = runs.iloc[0]
display(runs[["run_id", "tags.mlflow.runName", "metrics.rmse"]])

,run_id,tags.mlflow.runName,metrics.rmse
0,2f74fa55af7d47d5a00d67c95e48378a,rf-depth14,56.816898
1,a3cc0ffabd3e4776916fec78093f7382,rf-depth8,59.981820
2,b605a5e475bc4ff28acfc7263d5921dc,baseline-linear,94.003587


🖱️ **Pela interface:** **Experiments** → `aula01-experimentos` → **Training runs**. Na busca, digite
`tags.autor = 'turma'` e clique no cabeçalho da coluna `rmse` para ordenar.

## 2. Registrar versões

> 💬 No MLflow 3 aparece um aviso dizendo que o modelo foi encontrado como `models:/m-...`.
> É esperado: no MLflow 3 o modelo virou uma entidade própria (*LoggedModel*), ligada ao run.
>
> O `bike-demand` já tem versões: são os treinos do pipeline da T1 (o `train.py` registra cada treino).
> Por isso as versões novas não começam em 1. O código nunca depende do número: usa sempre `v.version`.

In [3]:
v_base = mlflow.register_model(f"runs:/{baseline.run_id}/model", MODEL_NAME)
v_melhor = mlflow.register_model(f"runs:/{melhor.run_id}/model", MODEL_NAME)

client.update_registered_model(MODEL_NAME, description="Previsão de demanda horária de bicicletas.")
client.update_model_version(MODEL_NAME, v_base.version,
                            description=f"Baseline da turma (T2): {baseline['tags.mlflow.runName']}")
client.update_model_version(MODEL_NAME, v_melhor.version,
                            description=f"Melhor run da turma na T2: {melhor['tags.mlflow.runName']}")

# Tags que identificam cada versão: de qual run veio, quem treinou e o RMSE de origem
for v, r in [(v_base, baseline), (v_melhor, melhor)]:
    client.set_model_version_tag(MODEL_NAME, v.version, "source_run_name", r["tags.mlflow.runName"])
    client.set_model_version_tag(MODEL_NAME, v.version, "autor", "turma")
    client.set_model_version_tag(MODEL_NAME, v.version, "test_rmse", f"{r['metrics.rmse']:.2f}")

client.set_model_version_tag(MODEL_NAME, v_base.version, "validation_status", "approved")
print("baseline -> versão", v_base.version, "| melhor run -> versão", v_melhor.version)

Registered model 'bike-demand' already exists. Creating a new version of this model...


2026/11/07 18:16:01 WARNING mlflow.tracking._model_registry.fluent: Run with id b605a5e475bc4ff28acfc7263d5921dc has no artifacts at artifact path 'model', registering model based on models:/m-9326a0de4763431c82bab1a6f50fe822 instead


2026/11/07 18:16:01 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: bike-demand, version 5


Created version '5' of model 'bike-demand'.


Registered model 'bike-demand' already exists. Creating a new version of this model...


2026/11/07 18:16:01 WARNING mlflow.tracking._model_registry.fluent: Run with id 2f74fa55af7d47d5a00d67c95e48378a has no artifacts at artifact path 'model', registering model based on models:/m-51d2953c154547698a742a0789a57971 instead


2026/11/07 18:16:01 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: bike-demand, version 6


Created version '6' of model 'bike-demand'.


baseline -> versão 5 | melhor run -> versão 6


🖱️ **Pela interface** (não repita agora: o código acima já fez):

- **Registrar:** abra o run → em **Logged models**, clique em `model` → botão **Register model** →
  escolha `bike-demand` → **Register**.
- **Descrição:** **Model registry** → `bike-demand` → clique na versão → **Description** → **Edit**.
- **Tag:** na mesma página, em **Tags**, preencha *Name* e *Value* e clique em **Add**.

Confira as duas versões novas em **Model registry** → `bike-demand`.

## 3. Primeiro *champion* e o consumidor

Imagine que o baseline foi aprovado e está em produção. A API da Aula 2 vai carregar
`models:/bike-demand@champion`: ela **não sabe** o número da versão, e isso é proposital.

In [4]:
client.set_registered_model_alias(MODEL_NAME, "champion", v_base.version)
print(client.get_registered_model(MODEL_NAME).aliases)
print(prever_com_champion(X_test.head(5)))

{'champion': '5'}


[56. 55. 42. 62. 46.]


🖱️ **Pela interface:** na página da versão, ao lado de **Aliases**, clique em **Add** (ou no lápis) →
digite `champion` → **Enter** → **Save aliases**. A coluna *Aliases* da lista de versões tem o mesmo atalho.

O consumidor, porém, só existe no código: é ele que carrega `models:/bike-demand@champion` e faz a previsão.

## 4. Um candidato (*challenger*) e a validação **no mesmo teste**

In [5]:
client.set_registered_model_alias(MODEL_NAME, "challenger", v_melhor.version)

m_champ = avaliar("champion")
m_chall = avaliar("challenger")
display(pd.DataFrame({"champion": m_champ, "challenger": m_chall}).T.round(3))

,rmse,mae,r2
champion,94.004,72.295,0.183
challenger,56.817,35.340,0.702


🖱️ **Pela interface:** o `challenger` é adicionado do mesmo jeito que o `champion` (passo 3). **A validação não dá para fazer na interface:**
é preciso carregar os dois modelos e prever no mesmo teste, como faz a função `avaliar`.
A interface mostra só o RMSE que foi gravado no run de origem (página da versão → **Source run**).

## 5. Promoção (guardando quem era o anterior, para permitir rollback)

In [6]:
if m_chall["rmse"] < m_champ["rmse"]:
    anterior = client.get_model_version_by_alias(MODEL_NAME, "champion").version
    client.set_model_version_tag(MODEL_NAME, v_melhor.version, "validation_status", "approved")
    client.set_model_version_tag(MODEL_NAME, v_melhor.version, "previous_champion", anterior)
    client.set_registered_model_alias(MODEL_NAME, "champion", v_melhor.version)
    client.delete_registered_model_alias(MODEL_NAME, "challenger")
    print(f"✅ Promovido: champion agora é a versão {v_melhor.version} (antes: {anterior})")
else:
    print("❌ O challenger não superou o champion. Nada muda.")

print(prever_com_champion(X_test.head(5)))   # o mesmo código, agora com outro modelo

✅ Promovido: champion agora é a versão 6 (antes: 5)


[64. 59. 37. 15.  5.]


🖱️ **Pela interface**, a promoção é feita na página da versão nova:

1. Em **Tags**, adicione `validation_status` = `approved` e `previous_champion` = número da versão que era champion.
2. Em **Aliases**, adicione `champion`. A interface avisa que o alias sai da versão antiga: cada alias aponta para uma versão só.
3. Ainda em **Aliases**, clique no **×** do `challenger` → **Save aliases**.

> ⚠️ O botão azul **Promote model** **não** faz isso: ele copia a versão para *outro* modelo registrado.
> Aqui a promoção é mover o alias.

Abra **Model registry** → `bike-demand` e veja as versões, os aliases e as tags.

## 6. 🚨 Incidente → rollback

A nova versão começou a dar previsões estranhas em produção. Precisamos voltar **agora**, sem novo deploy.

In [7]:
atual = client.get_model_version_by_alias(MODEL_NAME, "champion")
anterior = atual.tags["previous_champion"]

client.set_registered_model_alias(MODEL_NAME, "champion", anterior)
client.set_model_version_tag(MODEL_NAME, atual.version, "validation_status", "rolled_back")

print(f"↩️ Rollback: champion {atual.version} -> {anterior}")
print(prever_com_champion(X_test.head(5)))
display(resumo_registry())

↩️ Rollback: champion 6 -> 5


[56. 55. 42. 62. 46.]


,versão,aliases,origem,autor,status,descrição
5,1,,pipeline (T1),,,Treino do pipeline (dvc repro)
4,2,,pipeline (T1),,,Treino do pipeline (dvc repro)
3,3,,pipeline (T1),,,Treino do pipeline (dvc repro)
2,4,,pipeline (T1),,,Treino do pipeline (dvc repro)
1,5,champion,baseline-linear,turma,approved,Baseline da turma (T2): baseline-linear
0,6,,rf-depth14,turma,rolled_back,Melhor run da turma na T2: rf-depth14


🖱️ **Pela interface:** na versão anterior (veja o número na tag `previous_champion`), adicione o alias `champion`.
Na versão que saiu, edite a tag `validation_status` para `rolled_back` (lápis na linha da tag).

Repare: o consumidor (`prever_com_champion`) **não mudou uma linha**. Promoção e rollback são só
a troca de um ponteiro. Essa é a ideia central do Model Registry.

---
# Parte 2 · Sozinhos

O mesmo ciclo, agora com **o seu melhor run da T2**: registrar → *challenger* → validar → promover.
Registro e validação são por código. A promoção você faz por código **ou** pela interface.

In [8]:
meus = mlflow.search_runs(experiment_names=["aula01-experimentos"],
                          filter_string="tags.autor = 'aluno' and metrics.rmse > 0",
                          order_by=["metrics.rmse ASC"])
meu_melhor = meus.iloc[0]
print(meu_melhor["tags.mlflow.runName"], "RMSE", round(meu_melhor["metrics.rmse"], 1))

hgb-lr0.1-it300-leaves15 RMSE 50.7


### 1. Registre o seu melhor run como uma nova versão (como no passo 2)

In [9]:
v_meu = mlflow.register_model(f"runs:/{meu_melhor.run_id}/model", MODEL_NAME)
nome_run = meu_melhor["tags.mlflow.runName"]          # ex.: hgb-lr0.1-it300-leaves15
client.update_model_version(MODEL_NAME, v_meu.version, description=f"Meu melhor run da T2: {nome_run}")
client.set_model_version_tag(MODEL_NAME, v_meu.version, "source_run_name", nome_run)
client.set_model_version_tag(MODEL_NAME, v_meu.version, "autor", "aluno")
client.set_model_version_tag(MODEL_NAME, v_meu.version, "test_rmse", f"{meu_melhor['metrics.rmse']:.2f}")
print("sua versão:", v_meu.version, "|", nome_run)

Registered model 'bike-demand' already exists. Creating a new version of this model...


2026/11/07 18:16:09 WARNING mlflow.tracking._model_registry.fluent: Run with id 9dbd43b2170e4ad0bc800d95bdf55d13 has no artifacts at artifact path 'model', registering model based on models:/m-e2761fcb2fb3414b9d2271b8bcaf55e5 instead


2026/11/07 18:16:09 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: bike-demand, version 7


Created version '7' of model 'bike-demand'.


sua versão: 7 | hgb-lr0.1-it300-leaves15


### 2. Aponte o `@challenger` para ela e valide contra o champion atual (como no passo 4)

In [10]:
client.set_registered_model_alias(MODEL_NAME, "challenger", v_meu.version)
m_champ = avaliar("champion")
m_chall = avaliar("challenger")
display(pd.DataFrame({"champion": m_champ, "challenger": m_chall}).T.round(3))

,rmse,mae,r2
champion,94.004,72.295,0.183
challenger,50.715,33.931,0.762


### 3. Se o seu modelo venceu, promova, guardando o `previous_champion` (como no passo 5)

Escolha **um** caminho:
- **Código:** complete e rode a célula abaixo.
- **Interface:** siga o 🖱️ do passo 5, na sua versão, e **pule** a célula abaixo.

> ⚠️ Não faça pelos dois: se a célula rodar depois da promoção pela interface, a sua versão já é o champion
> e o `previous_champion` vira ela mesma.

In [11]:
if m_chall["rmse"] < m_champ["rmse"]:
    anterior = client.get_model_version_by_alias(MODEL_NAME, "champion").version
    client.set_model_version_tag(MODEL_NAME, v_meu.version, "validation_status", "approved")
    client.set_model_version_tag(MODEL_NAME, v_meu.version, "previous_champion", anterior)
    client.set_model_version_tag(MODEL_NAME, v_meu.version, "promovido_via", "codigo")
    client.set_registered_model_alias(MODEL_NAME, "champion", v_meu.version)
    client.delete_registered_model_alias(MODEL_NAME, "challenger")
    print(f"✅ Promovido: champion agora é a versão {v_meu.version} (antes: {anterior})")
else:
    print("❌ Não superou o champion. Nada muda.")

✅ Promovido: champion agora é a versão 7 (antes: 5)


### 4. Confira o Registry (pelos dois caminhos: tabela abaixo e **Model registry** → `bike-demand`)

In [12]:
display(resumo_registry())

,versão,aliases,origem,autor,status,descrição
6,1,,pipeline (T1),,,Treino do pipeline (dvc repro)
5,2,,pipeline (T1),,,Treino do pipeline (dvc repro)
4,3,,pipeline (T1),,,Treino do pipeline (dvc repro)
3,4,,pipeline (T1),,,Treino do pipeline (dvc repro)
2,5,,baseline-linear,turma,approved,Baseline da turma (T2): baseline-linear
1,6,,rf-depth14,turma,rolled_back,Melhor run da turma na T2: rf-depth14
0,7,champion,hgb-lr0.1-it300-leaves15,aluno,approved,Meu melhor run da T2: hgb-lr0.1-it300-leaves15


---
## Para discutir
1. Por que o consumidor (a API) usa o **alias** e não o número da versão?
2. No rollback, o que mudou no Registry? E no código do consumidor?
3. Por que champion e challenger precisam ser avaliados **no mesmo teste**?
4. Quase tudo do Registry dá para fazer na interface. O que não dá? E por que, em produção, a promoção costuma ser feita por código?

**Respostas esperadas:**
1. Para trocar o modelo em produção sem mudar nem reimplantar a API: promover é mover o ponteiro.
2. No Registry, só o alias `@champion` mudou de versão (e a tag `rolled_back`). No consumidor, nada.
3. Métricas medidas em períodos diferentes não são comparáveis: um período pode ser mais difícil que outro.
4. A validação (carregar os modelos e prever no mesmo teste) e o consumo pelo alias só existem no código.
   Em produção a promoção roda num pipeline (CI/CD, Aula 2), sem ninguém clicando, e o código guarda a regra
   da decisão (o `if` do RMSE). Na interface, o critério fica na cabeça de quem clicou. A interface é ótima para
   conferir e para um rollback de emergência.

## Resultado

| Versão | Origem | Autor | RMSE (teste) | Status final |
|---|---|---|---|---|
| 1–4 | pipeline da T1 (`dvc repro`) | — | — | sem alias |
| 5 | `baseline-linear` | turma | 94.0 | approved |
| 6 | `rf-depth14` | turma | 56.8 | rolled_back (incidente simulado) |
| **7** | **`hgb-lr0.1-it300-leaves15`** | **aluno** | **50.7** | **approved · @champion** |

Linha do tempo do `@champion`: **5** (baseline) → **6** (promoção) → **5** (rollback) → **7** (minha promoção).
O consumidor `prever_com_champion` não mudou uma linha em nenhuma dessas trocas.

Cada versão leva tags que dizem de onde veio: `source_run_name`, `autor`, `test_rmse`
(e `promovido_via` na promoção feita por código).

**Observações**
- A versão 7 e a versão 4 do pipeline são o **mesmo modelo** (mesmos dados, parâmetros e seed, RMSE 50.7),
  registrados por caminhos diferentes. O Registry não detecta duplicatas: cada registro é uma versão nova.
- Rodar o notebook de novo cria **mais versões** (8, 9…): o `register_model` não é idempotente.
- O rollback depende da tag `previous_champion`. Se o champion não tiver essa tag, `atual.tags["previous_champion"]`
  quebra com `KeyError`: por isso a promoção precisa sempre gravar quem era o anterior.
- O champion e o challenger são comparados no **teste**, o mesmo conjunto usado para escolher o melhor run na T2
  (a pegadinha da T1 de novo): a comparação favorece o challenger.